# 03 — Review Embeddings → Milvus

`store.materialize()` → Spark k8s:// GPU executors → sentence-transformers → Milvus

In [ ]:
%pip install -q pymilvus sentence-transformers s3fs pyarrow yamlmagic
%load_ext yamlmagic

Note: you may need to restart the kernel to use updated packages.


## Parameters

In [ ]:
%%yaml parameters

# Materialization window
MATERIALIZE_START: "2020-01-01T00:00:00"

# Milvus collection
COLLECTION_NAME: smartshop_review_embeddings

# Spark executor topology
EXECUTOR_INSTANCES: 4
EXECUTOR_CORES: 4
EXECUTOR_MEMORY: "16g"

# JDK path (workbench-specific)
JAVA_HOME: /opt/app-root/src/.local/java/jdk-17.0.2

Driver: 10.131.4.42  |  Spark master: k8s://https://kubernetes.default.svc:443
Executor image: image-registry.openshift-image-registry.svc:5000/smartshop/feast-spark-executor-embeddings:latest
Materialize: 2020-01-01 00:00:00+00:00 → 2026-05-04 07:30:00+00:00


In [ ]:
from _config import *
globals().update(parameters)
import socket
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_DIR = Path(os.environ.get("PWD", "/opt/app-root/src"))
FEAST_FEATURE_REPO = NOTEBOOK_DIR / "feature_repo"
FEAST_REPO_DIR = Path("/tmp/feast-milvus-repo")

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = f"{JAVA_HOME}/bin:" + os.environ.get("PATH", "")
os.environ["JAVA_TOOL_OPTIONS"] = "-Dcom.redhat.fips=false"

MY_POD_IP = socket.gethostbyname(socket.gethostname())
K8S_MASTER = "k8s://https://kubernetes.default.svc:443"
EXECUTOR_IMAGE = f"{INTERNAL_REGISTRY}/{NAMESPACE}/feast-spark-executor-embeddings:latest"

MAT_START = datetime.fromisoformat(MATERIALIZE_START).replace(tzinfo=timezone.utc)
MAT_END = datetime.now(timezone.utc)

validate()
print(f"Driver: {MY_POD_IP}  |  Spark master: {K8S_MASTER}")
print(f"Executor image: {EXECUTOR_IMAGE}")
print(f"Materialize: {MAT_START} → {MAT_END}")

## Pre-flight

In [ ]:
import warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

import base64
from kubernetes import client, config
config.load_incluster_config()
v1 = client.CoreV1Api()

checks = {}

for fname in ("feature_store_milvus.yaml", "features_milvus.py"):
    path = FEAST_FEATURE_REPO / fname
    checks[f"feature_repo/{fname}"] = "✓" if path.exists() else f"✗ not found at {path}"

# Load S3 credentials
try:
    secret = v1.read_namespaced_secret("smartshop-credentials", NAMESPACE)
    for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL_S3"):
        if key in secret.data:
            os.environ[key] = base64.b64decode(secret.data[key]).decode()
    checks["Secret smartshop-credentials"] = "✓"
except Exception as e:
    checks["Secret smartshop-credentials"] = f"✗ {e}"

# Fetch registry TLS cert
try:
    tls_cm = v1.read_namespaced_config_map("smartshop-feast-registry-tls", NAMESPACE)
    tls_dir = Path("/tmp/feast-tls")
    tls_dir.mkdir(exist_ok=True)
    (tls_dir / "tls.crt").write_text(tls_cm.data["tls.crt"])
    checks["Registry TLS cert"] = "✓"
except Exception as e:
    checks["Registry TLS cert"] = f"✗ {e}"

# Check JDK
checks["JDK"] = "✓" if Path(JAVA_HOME, "bin/java").exists() else f"✗ not at {JAVA_HOME}"

# Check Milvus
try:
    from pymilvus import MilvusClient
    mc = MilvusClient(uri=f"http://milvus.{NAMESPACE}.svc.cluster.local:19530")
    mc.list_collections()
    checks["Milvus connectivity"] = "✓"
except Exception as e:
    checks["Milvus connectivity"] = f"✗ {e}"

# Verify executor image exists in internal registry
try:
    apps_v1 = client.AppsV1Api()
    custom = client.CustomObjectsApi()
    is_obj = custom.get_namespaced_custom_object(
        "image.openshift.io", "v1", NAMESPACE,
        "imagestreams", "feast-spark-executor-embeddings",
    )
    tags = [t["tag"] for t in is_obj.get("status", {}).get("tags", [])]
    checks["Executor image"] = f"✓ tags: {tags}" if tags else "✗ no tags (build pending)"
except Exception as e:
    checks["Executor image"] = f"✗ {e}"

# Verify spark ServiceAccount exists (needed for executor pod creation)
try:
    v1.read_namespaced_service_account("spark", NAMESPACE)
    checks["SA spark"] = "✓"
except Exception as e:
    checks["SA spark"] = f"✗ {e}"

print("Pre-flight checks:")
print("-" * 70)
for k, v in checks.items():
    print(f"  {k:45s} {v}")

Pre-flight checks:
----------------------------------------------------------------------
  feature_repo/feature_store_milvus.yaml        ✓
  feature_repo/features_milvus.py                ✓
  Secret smartshop-credentials                   ✓
  Registry TLS cert                              ✓
  JDK                                            ✓
  Milvus connectivity                            ✓
  Executor image                                 ✓ tags: ['latest']
  SA spark                                       ✓


## Prepare Feast repo

In [ ]:
import shutil
import string
import yaml

# Create clean rendered repo
if FEAST_REPO_DIR.exists():
    shutil.rmtree(FEAST_REPO_DIR)
FEAST_REPO_DIR.mkdir(parents=True)

# Copy feature definitions (only milvus slice)
shutil.copy(FEAST_FEATURE_REPO / "features_milvus.py", FEAST_REPO_DIR / "features_milvus.py")

# Render feature_store_milvus.yaml with k8s:// distributed mode variables
raw_yaml = (FEAST_FEATURE_REPO / "feature_store_milvus.yaml").read_text()
rendered = string.Template(raw_yaml).safe_substitute(
    NAMESPACE=NAMESPACE,
    SPARK_MASTER=K8S_MASTER,
    MY_POD_IP=MY_POD_IP,
    SPARK_EXECUTOR_EMBEDDINGS_IMAGE=EXECUTOR_IMAGE,
)

cfg = yaml.safe_load(rendered)

# Restructure batch_engine: move flat spark.* keys into spark_conf sub-dict
# (SparkComputeEngineConfig Pydantic model only accepts: type, partitions, spark_conf)
be = cfg["batch_engine"]
spark_conf = {}
for key in list(be.keys()):
    if key.startswith("spark."):
        spark_conf[key] = be.pop(key)
    elif key not in ("type", "partitions"):
        be.pop(key)

# Driver S3 credentials — executors get theirs via secretKeyRef in the YAML
spark_conf["spark.hadoop.fs.s3a.access.key"] = os.environ["AWS_ACCESS_KEY_ID"]
spark_conf["spark.hadoop.fs.s3a.secret.key"] = os.environ["AWS_SECRET_ACCESS_KEY"]
spark_conf["spark.hadoop.fs.s3a.aws.credentials.provider"] = (
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)
be["spark_conf"] = spark_conf

# Same credentials for offline_store (used during feast apply schema validation)
os_conf = cfg["offline_store"]["spark_conf"]
os_conf["spark.hadoop.fs.s3a.access.key"] = os.environ["AWS_ACCESS_KEY_ID"]
os_conf["spark.hadoop.fs.s3a.secret.key"] = os.environ["AWS_SECRET_ACCESS_KEY"]
os_conf["spark.hadoop.fs.s3a.aws.credentials.provider"] = (
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

# Point TLS cert to local copy
cfg["registry"]["cert"] = str(Path("/tmp/feast-tls/tls.crt"))

# Write rendered config
(FEAST_REPO_DIR / "feature_store.yaml").write_text(yaml.dump(cfg, default_flow_style=False))

print(f"Repo: {FEAST_REPO_DIR}  |  {[f.name for f in FEAST_REPO_DIR.iterdir()]}")
print(f"Executors: {spark_conf['spark.executor.instances']} × {spark_conf['spark.executor.cores']} cores × {spark_conf['spark.executor.memory']}")

Repo: /tmp/feast-milvus-repo  |  ['features_milvus.py', 'feature_store.yaml']
Executors: 4 × 4 cores × 16g


## Feast apply

In [ ]:
import sys, time, importlib
from feast import FeatureStore

store = FeatureStore(repo_path=str(FEAST_REPO_DIR))

sys.path.insert(0, str(FEAST_REPO_DIR))
if "features_milvus" in sys.modules:
    del sys.modules["features_milvus"]
fm = importlib.import_module("features_milvus")
objects = [fm.review, fm.raw_reviews_source, fm.review_embeddings]

try:
    store.registry.delete_feature_view("review_embeddings", store.project)
except Exception:
    pass

t0 = time.time()
store.apply(objects)
print(f"feast apply: {time.time() - t0:.1f}s — review_embeddings registered ✓")

# Stop local[*] Spark session — materialize needs a fresh k8s:// session
from pyspark.sql import SparkSession
existing = SparkSession.getActiveSession()
if existing:
    existing.stop()
    print("Stopped local Spark session")

feast apply: 8.3s — review_embeddings registered ✓
Stopped local Spark session


## Materialize → Milvus

In [ ]:
t0 = time.time()
store.materialize(
    start_date=MAT_START,
    end_date=MAT_END,
    feature_views=["review_embeddings"],
)
print(f"\n✓ Materialized in {(time.time() - t0)/60:.1f}m")

Materializing 1 feature views from 2020-01-01 00:00:00+00:00 to 2026-05-04 07:30:00+00:00 into the milvus online store.

review_embeddings:
Connecting to Milvus remotely at http://milvus.smartshop.svc.cluster.local:19530
100%|██████████| 296/296 [42:18<00:00,  8.58s/it]

✓ Materialized in 43.2m


## Verify

In [ ]:
from pymilvus import MilvusClient

milvus = MilvusClient(uri=f"http://milvus.{NAMESPACE}.svc.cluster.local:19530")

if milvus.has_collection(COLLECTION_NAME):
    stats = milvus.get_collection_stats(COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}':")
    print(f"  Stats: {stats}")
    print(f"\n✓ Embeddings materialized into Milvus")
else:
    print(f"✗ Collection '{COLLECTION_NAME}' not found")

Collection 'smartshop_review_embeddings':
  Stats: {'row_count': 1048798}

✓ Embeddings materialized into Milvus


### Vector search test

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
query = "What do customers say about battery life?"
query_embedding = st_model.encode(query, normalize_embeddings=True).tolist()

# Serving config — no Spark needed, just registry + Milvus
import string as string_mod
serving_yaml = (FEAST_FEATURE_REPO / "feature_store_serving.yaml").read_text()
rendered_serving = string_mod.Template(serving_yaml).safe_substitute(NAMESPACE=NAMESPACE)
rendered_serving = rendered_serving.replace(
    "/var/run/secrets/feast-registry-tls/service-ca.crt",
    str(Path("/tmp/feast-tls/tls.crt")),
)
serving_path = Path("/tmp/feast-serving-repo")
serving_path.mkdir(exist_ok=True)
(serving_path / "feature_store.yaml").write_text(rendered_serving)
serving_store = FeatureStore(repo_path=str(serving_path))

t0 = time.time()
result = serving_store.retrieve_online_documents_v2(
    features=[
        "review_embeddings:embedding",
        "review_embeddings:embed_text",
        "review_embeddings:item_id",
        "review_embeddings:rating",
        "review_embeddings:review_title",
    ],
    query=query_embedding,
    top_k=5,
    distance_metric="COSINE",
)
latency = (time.time() - t0) * 1000
df = result.to_df()
print(f"Query: '{query}'  |  {latency:.0f}ms  |  {len(df)} results\n")

for i, row in df.iterrows():
    title = row.get("review_title", row.get("review_embeddings__review_title", ""))
    text = row.get("embed_text", row.get("review_embeddings__embed_text", ""))
    dist = row.get("distance", 0)
    print(f"{i+1}. [{dist:.3f}] {title}")
    print(f"   {str(text)[:150]}...")

Query: 'What do customers say about battery life?'  |  58ms  |  5 results

1. [0.396] Good battery life
   Good battery life I bought this for my son and he loves it. The battery life is great and it charges quickly. He uses it every day...
2. [0.403] great for a plane
   great for a plane again, great for a plane. I leave them behind me in hotels for the next guest. Noise cancelling is above averag...
3. [0.415] Long lasting battery
   Long lasting battery The battery on this device lasts forever. I only have to charge it once a week with heavy use. Very happy w...
4. [0.421] A Tremendous Bargain
   A Tremendous Bargain I thought these wouldn't be that good for 20 bucks but I was totally wrong. These are amazing. Above averag...
5. [0.428] Excellent product
   Excellent product Great sound quality and the battery lasts all day. Perfect for commuting and working out. Highly recommend thi...


**Next →** `04_serving.ipynb`